**STEP 1**:**INSTALL DEPENDENCIES**

In [1]:
!pip install faiss-cpu sentence-transformers rouge-score nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.3 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=3bf25969e6f9be1e0b8c0eb93400ba2552130c106314f8ec14818f81e83137aa
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


**STEP 2: IMPORT LIBRARIES**

In [4]:
import json
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import datetime
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer

**STEP 3:DATASET FILE CREATION**

In [2]:
data = """
{"text": "Photosynthesis is the process by which green plants make food using sunlight, carbon dioxide and water."}
{"text": "Chlorophyll is the green pigment present in leaves that helps in photosynthesis."}
{"text": "Plants release oxygen during photosynthesis."}
{"text": "Friction is the force that opposes motion between two surfaces."}
{"text": "Reproduction is the process by which living organisms produce new individuals."}
{"text": "Force is a push or pull acting on an object."}
{"text": "Gravity is the force by which the earth attracts objects towards itself."}
"""

with open("class8_science.jsonl", "w", encoding="utf-8") as f:
    f.write(data.strip())

print("Dataset created successfully.")

Dataset created successfully.


**STEP 4:LOAD DATASET**

In [5]:
chunks = []
with open('class8_science.jsonl',"r",encoding="utf-8") as f:
  for line in f:
    chunks.append(json.loads(line)['text'])

print("Loaded",len(chunks),"chunks")

Loaded 7 chunks


**STEP 5 :EMBEDDING**

In [6]:
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

embeddings = embed_model.encode(chunks)

print (embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(7, 384)


**STEP 6:BUILDING FAISS INDEX**

In [7]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index built successfully.")

FAISS index built successfully.


**STEP 7:RETRIEVAL FUNCTION**

In [8]:
def search(query,k=3):
  query_vec=embed_model.encode([query])
  D,I=index.search(np.array(query_vec),k)
  return [chunks[i] for i in I[0]], D[0]

**STEP 8:RELEVANCE CHECK**

In [9]:
def is_relevant(results):
  return results and len(results[0]) > 20

**STEP 9:GENERATE  STRICT CURRICULUM ANSWER**

In [10]:
def generate_answer(query):
    results, distances = search(query)

    # Higher distance = less relevant
    if distances[0] > 1.0:   # Adjust threshold if needed
        return "I’m focused on Class 8 Science. Please ask syllabus-based questions.", []

    answer = results[0]
    return answer, results

**STEP 10:LOGGING FUNCTION**

In [11]:
def log_interaction(query,answer):
  with open("log.txt","a",encoding="utf-8") as f:
    f.write(f"{datetime.datetime.now()}|query:{query}|answer:{answer}\n")

**STEP 11:SAMPLE PREDICTION**

In [12]:
query = "What is photosynthesis?"

answer, sources = generate_answer(query)

print("ANSWER:\n", answer)

print("\nSOURCES:")
for s in sources:
    print("-", s)

log_interaction(query, answer)

ANSWER:
 Photosynthesis is the process by which green plants make food using sunlight, carbon dioxide and water.

SOURCES:
- Photosynthesis is the process by which green plants make food using sunlight, carbon dioxide and water.
- Plants release oxygen during photosynthesis.
- Chlorophyll is the green pigment present in leaves that helps in photosynthesis.


**STEP 12:EVALUATION OF 15  TEST QUESTION**

In [13]:
test_cases = [
    ("What is photosynthesis?", "Photosynthesis is the process by which plants make food using sunlight."),

    ("What is chlorophyll?", "Chlorophyll is the green pigment present in leaves."),

    ("Why do plants need sunlight?", "Plants need sunlight for photosynthesis."),

    ("Why do plants release oxygen?", "Plants release oxygen during photosynthesis."),

    ("What is friction?", "Friction is the force that opposes motion."),

    ("Why is friction useful?", "Friction helps in movement and prevents slipping."),

    ("What is gravity?", "Gravity is the force by which Earth attracts objects."),

    ("What is force?", "Force is a push or pull acting on an object."),

    ("What is reproduction?", "Reproduction is the process by which living organisms produce offspring."),

    ("Why do leaves appear green?", "Leaves appear green because of chlorophyll."),

    ("What is the function of chlorophyll?", "Chlorophyll helps in photosynthesis."),

    ("What do plants absorb from air during photosynthesis?", "Plants absorb carbon dioxide from air."),

    ("What do plants absorb from soil?", "Plants absorb water and minerals from soil."),

    ("What is the role of roots in plants?", "Roots absorb water and minerals from soil."),

    ("Why is oxygen important for living organisms?", "Oxygen is important for respiration.")]

# Initialize the ROUGE scorer here
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

for question, reference in test_cases:
    generated, _ = generate_answer(question)

    bleu = sentence_bleu([reference.split()], generated.split())

    rouge = scorer.score(reference, generated)

    print("\nQUESTION:", question)
    print("BLEU:", round(bleu, 3))
    print("ROUGE-L:", round(rouge['rougeL'].fmeasure, 3))


QUESTION: What is photosynthesis?
BLEU: 0.458
ROUGE-L: 0.815

QUESTION: What is chlorophyll?
BLEU: 0.516
ROUGE-L: 0.8

QUESTION: Why do plants need sunlight?
BLEU: 0.0
ROUGE-L: 0.4

QUESTION: Why do plants release oxygen?
BLEU: 1.0
ROUGE-L: 1.0

QUESTION: What is friction?
BLEU: 0.517
ROUGE-L: 0.824

QUESTION: Why is friction useful?
BLEU: 0.0
ROUGE-L: 0.118

QUESTION: What is gravity?
BLEU: 0.434
ROUGE-L: 0.857

QUESTION: What is force?
BLEU: 1.0
ROUGE-L: 1.0

QUESTION: What is reproduction?
BLEU: 0.786
ROUGE-L: 0.857

QUESTION: Why do leaves appear green?
BLEU: 0.0
ROUGE-L: 0.111

QUESTION: What is the function of chlorophyll?
BLEU: 0.0
ROUGE-L: 0.5

QUESTION: What do plants absorb from air during photosynthesis?
BLEU: 0.0
ROUGE-L: 0.182

QUESTION: What do plants absorb from soil?
BLEU: 0
ROUGE-L: 0.0

QUESTION: What is the role of roots in plants?
BLEU: 0
ROUGE-L: 0.0

QUESTION: Why is oxygen important for living organisms?
BLEU: 0
ROUGE-L: 0.2


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

**STEP 13:OUT OF SYLLABUS QUERY TEST**

In [14]:
query = "Who is primeminister of india"
answer, sources = generate_answer(query)

print(answer)

I’m focused on Class 8 Science. Please ask syllabus-based questions.


**STEP 14:HUMAN REVIEW**

| Question | Answer Quality | Reviewer Comment |
|---------|---------------|------------------|
| What is photosynthesis? | Good | Accurate and textbook-based |
| What is chlorophyll? | Good | Correct and clearly explained |
| What is friction? | Good | Concise and relevant response |
| What is gravity? | Good | Accurate definition with proper wording |
| Prime Minister of India? | Good | Properly rejected as out-of-syllabus ||

**STEP 15: CONCLUSSION**


**The Curriculum-Based AI Tutor for Class 8 Science was successfully developed using a Retrieval-Augmented Generation (RAG) pipeline. The system uses sentence-transformer embeddings with FAISS-based semantic search to retrieve relevant textbook content and provide accurate, curriculum-focused answers.

The tutor restricts responses to Class 8 Science syllabus content, displays source snippets for transparency, and gracefully rejects out-of-syllabus queries using a fallback mechanism.

Model performance was evaluated using BLEU and ROUGE-L metrics across multiple test questions, along with manual human review. The results indicate that the system provides accurate and relevant textbook-based responses for syllabus-related queries.

This project demonstrates the practical implementation of semantic search, retrieval-based QA, and evaluation techniques in an educational AI application.**